In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

## Load the cleaned dataset

In [3]:
df=pd.read_csv(r"C:\Users\HP\Downloads\cafe_sales_data_analysis\Data\dirty_cafe_sales.xls")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


##  Basic Understanding of dataset

Q1 How many rows and columns are present

In [4]:
df.shape

(10000, 8)

Q2 what are the columns names

In [5]:
df.columns.tolist()

['Transaction ID',
 'Item',
 'Quantity',
 'Price Per Unit',
 'Total Spent',
 'Payment Method',
 'Location',
 'Transaction Date']

Q4. Are there missing values?

In [6]:
df.isnull().sum()


Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Q5. Are there duplicate transactions?

In [7]:
df.duplicated().sum()

np.int64(0)

## Check Numerical Statistics

In [8]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


## Check Categorical Columns

In [20]:
categorical_columns = df.select_dtypes(
    include="object"
).columns
for col in categorical_columns:
    print("\n",col)
    print(df[col].value_counts())


 Transaction ID
Transaction ID
TXN_9226047    1
TXN_8567525    1
TXN_4583012    1
TXN_6796890    1
TXN_9933628    1
              ..
TXN_3160411    1
TXN_7034554    1
TXN_4271903    1
TXN_4977031    1
TXN_1961373    1
Name: count, Length: 10000, dtype: int64

 Item
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
ERROR        292
Name: count, dtype: int64

 Quantity
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
Name: count, dtype: int64

 Price Per Unit
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
UNKNOWN     164
Name: count, dtype: int64

 Total Spent
Total Spent
6.0        979
12.0       939
3.0        930
4.0        923
20.0       746
15.0       734
8.0        677
10.0       524
2.0        497
9.0        479
5.0        4

##  Feature Engineering

In [21]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [22]:
df["Transaction Date"].dtype

dtype('<M8[ns]')

Feature 1 — Year

In [23]:
df["Year"] = df["Transaction Date"].dt.year

Feature 2 — Month Number

In [24]:
df["Month"] = df["Transaction Date"].dt.month

Feature 3 — Month Name

In [25]:
df["Month_Name"] = df["Transaction Date"].dt.month_name()

Feature 4 — Day

In [15]:
df["Day"] = df["Transaction Date"].dt.day

feature 5 — Day Name

In [16]:
df["Day_Name"] = df["Transaction Date"].dt.day_name()

Feature 6 — Revenue per Quantity

In [17]:
df["Revenue_Per_Item"] = (
    df["Total Spent"] / df["Quantity"]
)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

Feature 7 — Transaction Size

In [ ]:
df["Transaction_Category"] = pd.cut(
    df["Total Spent"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [ ]:
df["Transaction_Category"].value_counts()

## Check the New Dataset

In [ ]:
df.head()

In [ ]:
df.info()

 ### EDA
 

Q6. What is the total revenue?

In [ ]:
total_revenue = df["Total Spent"].sum()
print(total_revenue)

Q7. How many transactions were made?

In [ ]:
total_quantity = df["Quantity"].sum()
print(f"Total Quantity Sold: {total_quantity}")

Q9. What is the average transaction value?

In [ ]:
average_transaction = df["Total Spent"].mean()
print(f"Average Transaction Value: {average_transaction}")

Q10. What is the average quantity per transaction?

In [ ]:
average_quantity = df["Quantity"].mean()
print(f"Average Quantity per Transaction: {average_quantity}")

Q11. Which product generates the highest revenue?

In [ ]:
product_revenue = (df.groupby("Item")["Total Spent"].sum().sort_values(ascending=False))
print(product_revenue)

In [ ]:
top_product = product_revenue.idxmax()
top_product_revenue = product_revenue.max()
print("Top Product:", top_product)
print("Revenue:", top_product_revenue)

Graph — Revenue by Product

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=product_revenue.index,y=product_revenue.values)
plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Q12. Which product sells the highest quantity?

In [ ]:
product_quantity = (df.groupby("Item")["Quantity"].sum().sort_values(ascending=False))
print(product_quantity)
top_quantity_product = product_quantity.idxmax()
print("Product with highest quantity:",top_quantity_product)

Graph

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=product_quantity.index,y=product_quantity.values)
plt.title("Quantity Sold by Product")
plt.xlabel("Product")
plt.ylabel("Quantity Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Q13. What is the average price of each product?

In [ ]:
average_price = (df.groupby("Item")["Price Per Unit"].mean().sort_values(ascending=False))
average_price

Graph

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=average_price.index,y=average_price.values)
plt.title("Average Price per Product")
plt.xlabel("Product")
plt.ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Q14. Which payment method is most commonly used?

In [ ]:
payment_count = (df["Payment Method"].value_counts())
payment_count
most_used_payment = payment_count.idxmax()
print("Most used payment method:",most_used_payment)

Graph

In [ ]:
plt.figure(figsize=(5, 5))
plt.pie(
    payment_count.values,
    labels=payment_count.index,
    autopct="%1.1f%%",
)
plt.title("Payment Method Distribution")
plt.show()

Q15. Which payment method generates the most revenue?

In [ ]:
payment_revenue = (df.groupby("Payment Method")["Total Spent"].sum().sort_values(ascending=False))
payment_revenue
top_payment_revenue = payment_revenue.idxmax()
print("Highest revenue payment method:",top_payment_revenue)

Q16. Which location generates the highest revenue?

In [ ]:
location_revenue = (df.groupby("Location")["Total Spent"].sum().sort_values(ascending=False))
location_revenue
top_location = location_revenue.idxmax()
print("Highest revenue location:",
    top_location
)

In [ ]:
product_location = pd.pivot_table(df,values="Total Spent",index="Item",columns="Location",aggfunc="sum")
product_location

plt.figure(figsize=(10, 6))
sns.heatmap(product_location,annot=True,fmt=".0f")
plt.title("Product Revenue by Location")
plt.xlabel("Location")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

Q18 Is quantity related to total spending?

In [ ]:
correlation = df[["Quantity", "Price Per Unit", "Total Spent"]].corr()
correlation

plt.figure(figsize=(8, 6))
sns.heatmap(correlation,annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

Q20 Quantity vs Total Spent

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df,x="Quantity",y="Total Spent")
plt.title("Quantity vs Total Spent")
plt.xlabel("Quantity")
plt.ylabel("Total Spent")
plt.tight_layout()
plt.show()

 Q21 What are the top 5 products by revenue?

In [18]:
top_5_products = (df.groupby("Item")["Total Spent"].sum().sort_values(ascending=False).head(5))
top_5_products

Item
Tea        UNKNOWN3.07.54.53.06.06.01.51.57.51.57.53.03.0...
Cookie     ERROR5.05.02.02.01.02.05.03.03.05.01.04.03.05....
UNKNOWN    9.0ERROR5.012.025.015.015.012.020.04.012.012.0...
Juice      6.012.012.012.012.015.015.015.06.012.015.0UNKN...
Coffee     4.04.08.02.04.06.08.08.010.06.010.010.02.02.01...
Name: Total Spent, dtype: object

Q22 Transacton category count

In [19]:
category_count = (df["Transaction_Category"].value_counts())
category_count

KeyError: 'Transaction_Category'